# Modeling Dataset

Build historical features before adding outcome labels and temporal splits.


## Setup

Import shared libraries and define the dbt build helper. Run once per kernel session.


In [23]:
from pathlib import Path
import subprocess
import warnings
import json
import joblib

import numpy as np
import pandas as pd
from google.cloud import bigquery
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss
from sklearn.utils.validation import check_is_fitted

project_root = Path.cwd().resolve()
if not (project_root / "dbt_promo_abuse").is_dir():
    project_root = project_root.parent

dbt_project_dir = project_root / "dbt_promo_abuse"
dbt_executable = project_root / ".dbt-venv" / "bin" / "dbt"
assert (dbt_project_dir / "profiles.yml").exists(), "Project dbt profile not found."
assert dbt_executable.exists(), "Project dbt executable not found."


def build_dbt_models(*model_names):
    """Build selected models and tests supported by their dependencies."""
    if not model_names:
        raise ValueError("Provide at least one model name.")
    command = [
        str(dbt_executable),
        "build",
        "--project-dir", str(dbt_project_dir),
        "--profiles-dir", str(dbt_project_dir),
        "--indirect-selection", "buildable",
        "--select", *model_names,
    ]
    print("Building:", ", ".join(model_names), "| test selection: buildable", flush=True)
    result = subprocess.run(
        command, cwd=project_root, capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    result.check_returncode()
    return result

PROJECT_ID = "alibaba-promo-abuse-1790454859"


## Initial Behavioral Feature Table

Build and test user-week counts, acceleration, recency, active days, redemption ratio, and cross-channel indicators in BigQuery.


In [7]:
feature_build_result = build_dbt_models("user_behavior_features")


Building: user_behavior_features | test selection: buildable
23:18:52  Running with dbt=1.12.5
23:18:53  Registered adapter: bigquery=1.12.1
23:18:53  Found 11 models, 77 data tests, 3 sources, 561 macros
23:18:53  
23:18:53  Concurrency: 4 threads (target='dev')
23:18:53  
23:18:57  1 of 8 START sql table model promo_abuse_features.user_behavior_features ....... [RUN]
23:19:04  1 of 8 OK created sql table model promo_abuse_features.user_behavior_features .. [CREATE TABLE (723.5k rows, 248.4 MiB processed) in 6.48s]
23:19:04  2 of 8 START test accepted_values_user_behavior_features_both_click_redemption_accelerating__False__0__1  [RUN]
23:19:04  3 of 8 START test behavior_features_population_reconciliation .................. [RUN]
23:19:04  4 of 8 START test behavior_features_unique_grain ............................... [RUN]
23:19:04  5 of 8 START test behavior_features_value_checks ............................... [RUN]
23:19:05  2 of 8 PASS accepted_values_user_behavior_features_both

## Outcome Labels and Temporal Splits

Join seven-day outcome labels to historical features and apply the fixed temporal splits.


In [8]:
label_build_result = build_dbt_models(
    "user_outcome_labels", "user_modeling_dataset"
)


Building: user_outcome_labels, user_modeling_dataset | test selection: buildable
23:19:08  Running with dbt=1.12.5
23:19:09  Registered adapter: bigquery=1.12.1
23:19:09  Found 11 models, 77 data tests, 3 sources, 561 macros
23:19:09  
23:19:09  Concurrency: 4 threads (target='dev')
23:19:09  
23:19:12  1 of 18 START sql table model promo_abuse_features.user_outcome_labels ......... [RUN]
23:19:15  1 of 18 OK created sql table model promo_abuse_features.user_outcome_labels .... [CREATE TABLE (723.5k rows, 11.2 MiB processed) in 3.09s]
23:19:15  2 of 18 START test accepted_values_user_outcome_labels_abuse_next_7d__False__0__1  [RUN]
23:19:15  3 of 18 START test not_null_user_outcome_labels_abuse_next_7d .................. [RUN]
23:19:15  4 of 18 START test not_null_user_outcome_labels_prediction_date ................ [RUN]
23:19:15  5 of 18 START test not_null_user_outcome_labels_user_id ........................ [RUN]
23:19:16  2 of 18 PASS accepted_values_user_outcome_labels_abuse_next

## Training Data Readiness

Load training rows only and inspect predictor missingness, ranges, and constants before preprocessing.


In [11]:
client = bigquery.Client(project=PROJECT_ID)
training_sql = f"""
SELECT *
FROM `{PROJECT_ID}.promo_abuse_features.user_modeling_dataset`
WHERE dataset_split = 'train'
  AND prediction_date BETWEEN DATE '2016-02-01' AND DATE '2016-04-25'
"""

# Estimate scan size before issuing the training-only query.
max_bytes_billed = 1_000_000_000
training_dry_run = client.query(
    training_sql,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
print(f"Estimated scan: {training_dry_run.total_bytes_processed / 1_000_000:.1f} MB")
if training_dry_run.total_bytes_processed > max_bytes_billed:
    raise RuntimeError("Training query exceeds the 1 GB billing cap.")

training_job = client.query(
    training_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=max_bytes_billed),
)
train_df = training_job.result().to_dataframe(create_bqstorage_client=False)
print(f"Loaded {len(train_df):,} training user-weeks; query cache hit: {training_job.cache_hit}")


Estimated scan: 140.5 MB
Loaded 483,431 training user-weeks; query cache hit: False


In [12]:
assert len(train_df) == 483_431
assert train_df["abuse_next_7d"].isin([0, 1]).all()
assert train_df["abuse_next_7d"].sum() == 5_552
assert not train_df.duplicated(["user_id", "prediction_date"]).any()
assert train_df[["user_id", "prediction_date"]].notna().all().all()
assert train_df["dataset_split"].eq("train").all()
assert pd.to_datetime(train_df["prediction_date"]).between(
    "2016-02-01", "2016-04-25"
).all()

# Identifier, date, split, and target are never predictors.
metadata_columns = ["user_id", "prediction_date", "dataset_split", "abuse_next_7d"]
feature_columns = [c for c in train_df.columns if c not in metadata_columns]
assert all(pd.api.types.is_numeric_dtype(train_df[c]) for c in feature_columns)
X_train = train_df[feature_columns].astype("float64")
y_train = train_df["abuse_next_7d"].astype("int64")
assert not np.isinf(X_train.to_numpy()).any(), "Infinite predictor values found."

training_summary = pd.DataFrame([{
    "observations": len(train_df),
    "distinct_users": train_df["user_id"].nunique(),
    "positive_cases": int(y_train.sum()),
    "positive_rate_pct": y_train.mean() * 100,
    "predictors": len(feature_columns),
}])
feature_audit = pd.DataFrame({
    "missing_count": X_train.isna().sum(),
    "missing_pct": X_train.isna().mean() * 100,
    "unique_nonmissing": X_train.nunique(),
    "min": X_train.min(),
    "median": X_train.median(),
    "p99": X_train.quantile(0.99),
    "max": X_train.max(),
}).rename_axis("feature").sort_values("missing_pct", ascending=False)

expected_nullable = {
    "days_since_last_offline_purchase",
    "days_since_last_offline_redemption",
    "days_since_last_online_click",
    "offline_redemption_rate_30d",
}
unexpected_missing = [
    c for c in feature_columns
    if X_train[c].isna().any() and c not in expected_nullable
]
constant_features = feature_audit.index[
    feature_audit["unique_nonmissing"] <= 1
].tolist()

display(training_summary.round(3))
with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(feature_audit.round(3))
print("Unexpected missing-value columns:", unexpected_missing)
print("Constant or entirely missing columns:", constant_features)
assert not unexpected_missing, "Investigate unexpected missing values before modeling."


,observations,distinct_users,positive_cases,positive_rate_pct,predictors
0,483431,131004,5552,1.148,35


,missing_count,missing_pct,unique_nonmissing,min,median,p99,max
feature,,,,,,,
days_since_last_offline_redemption,423334,87.569,30,1.000,11.0,29.000,30.000
days_since_last_offline_purchase,343008,70.953,30,1.000,8.0,30.000,30.000
offline_redemption_rate_30d,236381,48.897,299,0.000,0.0,1.000,10.000
days_since_last_online_click,225893,46.727,30,1.000,9.0,29.000,30.000
offline_events_30d,0,0.000,85,0.000,1.0,22.000,150.000
online_active_days_7d,0,0.000,8,0.000,0.0,4.000,7.000
offline_coupon_receipts_acceleration,0,0.000,388,-3.130,0.0,1.000,4.814
offline_purchases_acceleration,0,0.000,347,-2.658,0.0,0.385,2.894
offline_redemptions_acceleration,0,0.000,225,-2.627,0.0,0.286,3.019


Unexpected missing-value columns: []
Constant or entirely missing columns: []


## Logistic Regression Baseline

Fit a regularized baseline with training-only scaling, explicit missing-value handling, and no class reweighting.


In [14]:
# Total-event counts duplicate the component counts in this dataset.
excluded_totals = [
    "offline_events_30d", "offline_events_7d",
    "online_events_30d", "online_events_7d",
]
baseline_feature_columns = [c for c in feature_columns if c not in excluded_totals]
recency_columns = [c for c in baseline_feature_columns if c.startswith("days_since_")]
ratio_columns = ["offline_redemption_rate_30d"]
complete_columns = [
    c for c in baseline_feature_columns
    if c not in recency_columns + ratio_columns
]
assert not X_train[complete_columns].isna().any().any()

# 31 is a sentinel for no relevant event within the 30-day history.
# Existing activity-presence indicators distinguish this from observed recency.
recency_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value=31)),
    ("scale", StandardScaler()),
])
# Keep undefined ratios distinguishable from observed ratios of zero.
ratio_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value=0, add_indicator=True)),
    ("scale", StandardScaler()),
])
preprocessor = ColumnTransformer([
    ("recency", recency_pipeline, recency_columns),
    ("ratio", ratio_pipeline, ratio_columns),
    ("complete", StandardScaler(), complete_columns),
], remainder="drop")

baseline_model = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(
        C=1.0,
        solver="lbfgs",
        max_iter=2000,
        class_weight=None,
    )),
])

# Fit preprocessing and the regularized model on training observations only.
with warnings.catch_warnings(record=True) as fit_warnings:
    warnings.simplefilter("always")
    baseline_model.fit(X_train[baseline_feature_columns], y_train)
for warning in fit_warnings:
    print(f"{warning.category.__name__}: {warning.message}")

convergence_warning = any(
    issubclass(w.category, ConvergenceWarning) for w in fit_warnings
)
fit_summary = pd.DataFrame([{
    "training_rows": len(X_train),
    "input_features": len(baseline_feature_columns),
    "transformed_features": len(
        baseline_model.named_steps["preprocess"].get_feature_names_out()
    ),
    "iterations": int(baseline_model.named_steps["model"].n_iter_.max()),
    "convergence_warning": convergence_warning,
}])
display(fit_summary)
assert not convergence_warning, "Resolve convergence before evaluating the baseline."


,training_rows,input_features,transformed_features,iterations,convergence_warning
0,483431,31,32,29,False


## Validation Performance

Score validation weeks only; compare ranking and probability quality with a constant training-prevalence baseline.


In [16]:
check_is_fitted(baseline_model.named_steps["model"])
assert not convergence_warning, "Resolve the training convergence warning first."

validation_sql = f"""
SELECT *
FROM `{PROJECT_ID}.promo_abuse_features.user_modeling_dataset`
WHERE dataset_split = 'validation'
  AND prediction_date BETWEEN DATE '2016-05-02' AND DATE '2016-05-16'
"""
validation_dry_run = client.query(
    validation_sql,
    job_config=bigquery.QueryJobConfig(dry_run=True, use_query_cache=False),
)
print(f"Estimated validation scan: {validation_dry_run.total_bytes_processed / 1_000_000:.1f} MB")
if validation_dry_run.total_bytes_processed > 1_000_000_000:
    raise RuntimeError("Validation query exceeds the 1 GB billing cap.")
validation_job = client.query(
    validation_sql,
    job_config=bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000),
)
validation_df = validation_job.result().to_dataframe(create_bqstorage_client=False)
assert not validation_df.duplicated(["user_id", "prediction_date"]).any()
assert validation_df[["user_id", "prediction_date"]].notna().all().all()
assert validation_df["dataset_split"].eq("validation").all()
assert validation_df["abuse_next_7d"].isin([0, 1]).all()
assert set(pd.to_datetime(validation_df["prediction_date"])) == set(
    pd.date_range("2016-05-02", "2016-05-16", freq="7D")
)
X_validation = validation_df[baseline_feature_columns].astype("float64")
y_validation = validation_df["abuse_next_7d"].astype("int64")
assert not np.isinf(X_validation.to_numpy()).any()
assert not X_validation[complete_columns].isna().any().any()

# Transform and predict only. Never refit preprocessing on validation data.
validation_scores = baseline_model.predict_proba(X_validation)[:, 1]
validation_predictions = validation_df[
    ["user_id", "prediction_date", "abuse_next_7d"]
].copy()
validation_predictions["risk_score"] = validation_scores
constant_scores = np.full(len(y_validation), y_train.mean())
metric_rows = []
for name, scores in [("Logistic regression", validation_scores),
                     ("Constant training prevalence", constant_scores)]:
    metric_rows.append({
        "model": name,
        "average_precision": average_precision_score(y_validation, scores)
            if y_validation.sum() > 0 else np.nan,
        "roc_auc": roc_auc_score(y_validation, scores)
            if y_validation.nunique() == 2 else np.nan,
        "brier_score": brier_score_loss(y_validation, scores),
        "log_loss": log_loss(y_validation, scores, labels=[0, 1]),
    })
validation_metrics = pd.DataFrame(metric_rows)
print(f"Validation rows: {len(y_validation):,}; positives: {y_validation.sum():,}; prevalence: {y_validation.mean():.3%}")
display(validation_metrics.round(5))


Estimated validation scan: 30.3 MB
Validation rows: 102,860; positives: 1,149; prevalence: 1.117%


,model,average_precision,roc_auc,brier_score,log_loss
0,Logistic regression,0.17376,0.90998,0.01017,0.04490
1,Constant training prevalence,0.01117,0.50000,0.01105,0.06132


## Weekly Review Capacity

Evaluate illustrative capacities of 100, 250, and 500 account reviews per week.


In [17]:
weekly_rows = []
for prediction_date, week in validation_predictions.groupby("prediction_date"):
    ranked = week.sort_values(
        ["risk_score", "user_id"], ascending=[False, True], kind="mergesort"
    )
    positives = int(week["abuse_next_7d"].sum())
    prevalence = week["abuse_next_7d"].mean()
    for capacity in [100, 250, 500]:
        selected = ranked.head(capacity)
        true_positives = int(selected["abuse_next_7d"].sum())
        precision = true_positives / len(selected)
        weekly_rows.append({
            "prediction_date": prediction_date,
            "capacity": capacity,
            "reviewed": len(selected),
            "weekly_positives": positives,
            "true_positives": true_positives,
            "false_positives": len(selected) - true_positives,
            "precision_at_k": precision,
            "recall_at_k": true_positives / positives if positives else np.nan,
            "lift_vs_weekly_baseline": precision / prevalence if prevalence else np.nan,
            "expected_random_positives": len(selected) * prevalence,
            "lowest_selected_score": selected["risk_score"].min(),
        })
weekly_capacity_results = pd.DataFrame(weekly_rows)
capacity_summary = weekly_capacity_results.groupby("capacity").agg(
    reviewed=("reviewed", "sum"),
    true_positives=("true_positives", "sum"),
    false_positives=("false_positives", "sum"),
    total_positives=("weekly_positives", "sum"),
    expected_random_positives=("expected_random_positives", "sum"),
)
capacity_summary["precision_at_k"] = capacity_summary["true_positives"] / capacity_summary["reviewed"]
capacity_summary["recall_at_k"] = capacity_summary["true_positives"] / capacity_summary["total_positives"].replace(0, np.nan)
capacity_summary["lift_vs_random_review"] = capacity_summary["true_positives"] / capacity_summary["expected_random_positives"].replace(0, np.nan)
display(weekly_capacity_results.round(4))
display(capacity_summary.round(4))


,prediction_date,capacity,reviewed,weekly_positives,true_positives,false_positives,precision_at_k,recall_at_k,lift_vs_weekly_baseline,expected_random_positives,lowest_selected_score
0,2016-05-02,100,100,371,33,67,0.330,0.0889,29.5675,1.1161,0.3816
1,2016-05-02,250,250,371,60,190,0.240,0.1617,21.5036,2.7902,0.1999
2,2016-05-02,500,500,371,93,407,0.186,0.2507,16.6653,5.5805,0.1148
3,2016-05-09,100,100,385,42,58,0.420,0.1091,37.5796,1.1176,0.4193
4,2016-05-09,250,250,385,75,175,0.300,0.1948,26.8426,2.7941,0.2040
5,2016-05-09,500,500,385,113,387,0.226,0.2935,20.2214,5.5881,0.1206
6,2016-05-16,100,100,393,37,63,0.370,0.0941,33.1126,1.1174,0.4095
7,2016-05-16,250,250,393,77,173,0.308,0.1959,27.5640,2.7935,0.2265
8,2016-05-16,500,500,393,116,384,0.232,0.2952,20.7625,5.5870,0.1316


,reviewed,true_positives,false_positives,total_positives,expected_random_positives,precision_at_k,recall_at_k,lift_vs_random_review
capacity,,,,,,,,
100,300,112,188,1149,3.3511,0.3733,0.0975,33.4217
250,750,212,538,1149,8.3778,0.2827,0.1845,25.3050
500,1500,322,1178,1149,16.7556,0.2147,0.2802,19.2175


## Calibration and Model Coefficients

Compare predicted and observed rates by score decile and inspect the largest fitted coefficients.


In [18]:
calibration_input = validation_predictions.copy()
calibration_input["score_bin"] = pd.qcut(
    calibration_input["risk_score"], q=10, duplicates="drop"
)
calibration_table = calibration_input.groupby("score_bin", observed=True).agg(
    observations=("abuse_next_7d", "size"),
    positive_cases=("abuse_next_7d", "sum"),
    mean_predicted_risk=("risk_score", "mean"),
    observed_positive_rate=("abuse_next_7d", "mean"),
    minimum_score=("risk_score", "min"),
    maximum_score=("risk_score", "max"),
)
calibration_table["prediction_minus_observed_pp"] = (
    calibration_table["mean_predicted_risk"] - calibration_table["observed_positive_rate"]
) * 100
coefficient_table = pd.DataFrame({
    "feature": baseline_model.named_steps["preprocess"].get_feature_names_out(),
    "coefficient": baseline_model.named_steps["model"].coef_[0],
})
coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values("absolute_coefficient", ascending=False)
display(calibration_table.round(5))
display(coefficient_table.head(15).round(4))
print("Coefficient signs describe conditional model associations, not causal effects.")
print("Correlated counts, acceleration, and indicators can make individual coefficients unstable.")


,observations,positive_cases,mean_predicted_risk,observed_positive_rate,minimum_score,maximum_score,prediction_minus_observed_pp
score_bin,,,,,,,
"(-0.001, 0.000315]",10286,6,0.00020,0.00058,0.00000,0.00032,-0.03809
"(0.000315, 0.000495]",10287,3,0.00040,0.00029,0.00032,0.00050,0.01099
"(0.000495, 0.000818]",10311,5,0.00064,0.00048,0.00050,0.00082,0.01557
"(0.000818, 0.00147]",10260,9,0.00110,0.00088,0.00082,0.00147,0.02275
"(0.00147, 0.00262]",10286,11,0.00200,0.00107,0.00147,0.00262,0.09316
"(0.00262, 0.00442]",10290,17,0.00344,0.00165,0.00262,0.00442,0.17915
"(0.00442, 0.0079]",10282,35,0.00597,0.0034,0.00442,0.00790,0.25631
"(0.0079, 0.0138]",10286,67,0.01063,0.00651,0.00790,0.01381,0.41152
"(0.0138, 0.0272]",10286,163,0.01922,0.01585,0.01381,0.02719,0.33698


,feature,coefficient,absolute_coefficient
11,complete__offline_active_days_30d,1.6988,1.6988
7,complete__offline_purchases_30d,-1.2994,1.2994
2,recency__days_since_last_online_click,-1.1161,1.1161
17,complete__online_clicks_30d,-0.7810,0.7810
1,recency__days_since_last_offline_redemption,-0.6941,0.6941
20,complete__online_active_days_7d,0.6854,0.6854
8,complete__offline_purchases_7d,-0.6336,0.6336
18,complete__online_clicks_7d,-0.6164,0.6164
27,complete__has_offline_redemption,0.5519,0.5519
19,complete__online_active_days_30d,0.4786,0.4786


Coefficient signs describe conditional model associations, not causal effects.
Correlated counts, acceleration, and indicators can make individual coefficients unstable.


## High-Score Calibration

Inspect narrower probability bands to check whether the highest scores are overconfident.


In [19]:
tail_input = validation_predictions.copy()
tail_input["risk_band"] = pd.cut(
    tail_input["risk_score"],
    bins=[0, 0.01, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.0],
    include_lowest=True,
)
tail_calibration = tail_input.groupby("risk_band", observed=True).agg(
    observations=("abuse_next_7d", "size"),
    positives=("abuse_next_7d", "sum"),
    mean_predicted=("risk_score", "mean"),
    observed_rate=("abuse_next_7d", "mean"),
)
tail_calibration["overprediction_pp"] = (
    tail_calibration["mean_predicted"] - tail_calibration["observed_rate"]
) * 100
display(tail_calibration.round(4))


,observations,positives,mean_predicted,observed_rate,overprediction_pp
risk_band,,,,,
"(-0.001, 0.01]",76217,106,0.0024,0.0014,0.096
"(0.01, 0.05]",21728,421,0.0213,0.0194,0.1968
"(0.05, 0.1]",2913,241,0.0687,0.0827,-1.4002
"(0.1, 0.2]",1220,166,0.1358,0.1361,-0.0252
"(0.2, 0.4]",480,100,0.2783,0.2083,6.9939
"(0.4, 0.6]",167,49,0.4906,0.2934,19.7213
"(0.6, 0.8]",91,42,0.6987,0.4615,23.7198
"(0.8, 1.0]",44,24,0.8964,0.5455,35.0916


## Baseline Refinement Comparison

Compare four regularization strengths and one log-count variant on the same validation weeks. All fitting uses training data only.


In [22]:
# Smaller C applies stronger L2 regularization.
# Reuse the fitted C=1 baseline; train only the four new candidates.
candidate_models = {"raw_C1": baseline_model}
for c_value in [0.01, 0.1, 10.0]:
    candidate_models[f"raw_C{c_value:g}"] = clone(baseline_model).set_params(
        model__C=c_value
    )

log_count_columns = [
    c for c in baseline_feature_columns
    if c.endswith(("_7d", "_30d"))
    and "active_days" not in c
    and c != "offline_redemption_rate_30d"
]
assert (X_train[log_count_columns] >= 0).all().all()
assert (X_validation[log_count_columns] >= 0).all().all()


def log_activity_counts(frame):
    transformed = frame.copy()
    transformed[log_count_columns] = np.log1p(transformed[log_count_columns])
    return transformed


# Leave acceleration, recency, ratios, and binary indicators unchanged.
candidate_models["log_counts_C1"] = Pipeline([
    ("log_counts", FunctionTransformer(log_activity_counts, validate=False)),
    ("preprocess", clone(baseline_model.named_steps["preprocess"])),
    ("model", clone(baseline_model.named_steps["model"])),
])

refinement_rows = []
candidate_validation_scores = {}
for name, candidate in candidate_models.items():
    print(f"Evaluating {name}...", flush=True)
    captured_warnings = []
    if name != "raw_C1":
        with warnings.catch_warnings(record=True) as captured_warnings:
            warnings.simplefilter("always")
            candidate.fit(X_train[baseline_feature_columns], y_train)
    for warning in captured_warnings:
        print(f"{warning.category.__name__}: {warning.message}")
    converged = not any(
        issubclass(w.category, ConvergenceWarning) for w in captured_warnings
    )
    scores = candidate.predict_proba(X_validation)[:, 1]
    candidate_validation_scores[name] = scores
    row = {
        "candidate": name,
        "converged": converged,
        "iterations": int(candidate.named_steps["model"].n_iter_.max()),
        "average_precision": average_precision_score(y_validation, scores),
        "roc_auc": roc_auc_score(y_validation, scores),
        "brier_score": brier_score_loss(y_validation, scores),
        "log_loss": log_loss(y_validation, scores, labels=[0, 1]),
    }
    ranked = validation_predictions.assign(candidate_score=scores).sort_values(
        ["prediction_date", "candidate_score", "user_id"],
        ascending=[True, False, True], kind="mergesort",
    )
    for k in [100, 250, 500]:
        reviewed = ranked.groupby("prediction_date", sort=False).head(k)
        tp = reviewed["abuse_next_7d"].sum()
        row[f"precision_at_{k}"] = tp / len(reviewed)
        row[f"recall_at_{k}"] = tp / y_validation.sum()
    refinement_rows.append(row)

refinement_results = pd.DataFrame(refinement_rows).sort_values(
    "average_precision", ascending=False
)
refinement_results["ap_change_vs_baseline"] = (
    refinement_results["average_precision"]
    - average_precision_score(y_validation, validation_scores)
)
with pd.option_context("display.max_columns", None):
    display(refinement_results.round(5))
print("Validation comparisons guide selection; they are not final held-out performance.")
print("Review convergence and weekly-capacity tradeoffs before choosing a candidate.")


Evaluating raw_C1...
Evaluating raw_C0.01...
Evaluating raw_C0.1...
Evaluating raw_C10...
Evaluating log_counts_C1...


,candidate,converged,iterations,average_precision,roc_auc,brier_score,log_loss,precision_at_100,recall_at_100,precision_at_250,recall_at_250,precision_at_500,recall_at_500,ap_change_vs_baseline
3,raw_C10,True,29,0.17376,0.90995,0.01017,0.04491,0.37333,0.09748,0.28267,0.18451,0.21467,0.28024,0.00000
0,raw_C1,True,29,0.17376,0.90998,0.01017,0.04490,0.37333,0.09748,0.28267,0.18451,0.21467,0.28024,0.00000
2,raw_C0.1,True,25,0.17370,0.90983,0.01016,0.04488,0.37333,0.09748,0.28000,0.18277,0.21267,0.27763,-0.00005
1,raw_C0.01,True,22,0.17078,0.91051,0.01014,0.04486,0.37000,0.09661,0.27333,0.17842,0.20933,0.27328,-0.00298
4,log_counts_C1,True,35,0.15714,0.91422,0.01030,0.04478,0.33333,0.08703,0.26000,0.16971,0.19533,0.25500,-0.01662


Validation comparisons guide selection; they are not final held-out performance.
Review convergence and weekly-capacity tradeoffs before choosing a candidate.


## Save Model-Comparison Inputs

Save training/validation data and the fitted logistic baseline locally for the next notebook.


In [24]:
# Export only training/validation data and the original fitted logistic baseline.
assert train_df["dataset_split"].eq("train").all()
assert validation_df["dataset_split"].eq("validation").all()
check_is_fitted(baseline_model.named_steps["model"])
comparison_dir = project_root / "data" / "processed" / "model_comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(comparison_dir / "train.parquet", index=False)
validation_df.to_parquet(comparison_dir / "validation.parquet", index=False)
joblib.dump(baseline_model, comparison_dir / "logistic_baseline.joblib")
(comparison_dir / "feature_columns.json").write_text(
    json.dumps(baseline_feature_columns, indent=2) + "\n"
)
print(f"Saved {len(train_df):,} training and {len(validation_df):,} validation rows.")
print("Saved comparison inputs to:", comparison_dir)


Saved 483,431 training and 102,860 validation rows.
Saved comparison inputs to: /Users/andrewlu/Desktop/alibaba-promo-abuse/data/processed/model_comparison
